# M9.18 · Arka plan çıkarma — fark, hareketli ortalama, GMM, ViBe

Her pikseli ön plan ya da arka plan olarak sınıflamak: kare farkı, üstel hareketli ortalama, piksel başına Gauss karışımı (Stauffer–Grimson) ve parametresiz, rastgele güncellemeli ViBe. Saf Python'da ViBe sınıflandırması ve hareketli ortalama; NumPy'da tüm görüntü için ViBe (komşulardan başlatma, hafızasız rastgele yer değiştirme, uzamsal yayılım); PyTorch'ta piksel başına K Gauss'luk karışım modeli ve OpenCV MOG2 ile karşılaştırma; en sonda üstel ortalamanın doğrusal bir özyineleme olduğunu fark edip zaman döngüsü olmadan, parçalı taramayla hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-18-background.html](https://m-gonul.github.io/pytorch-lab/m9-18-background.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_18_background.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"torch\"}))\n    rng = np.random.default_rng(0)\n\n    def vc():\n        ensure(need(ns, \"vibe_classify\")([[[100]], [[130]], [[95]], [[180]], [[140]]], [[110]]) == [[0]], \"elle örnek: arka plan\")\n        S = rng.integers(0, 256, (20, 12, 15)); f = rng.integers(0, 256, (12, 15))\n        got = np.array(ns[\"vibe_classify\"](S.tolist(), f.tolist(), 20, 2))\n        want = ((np.abs(S - f) < 20).sum(0) < 2).astype(int)\n        ensure(np.array_equal(got, want), \"rastgele model\")\n    R.test(\"vibe_classify\", vc)\n\n    def ra():\n        F = rng.integers(0, 256, (6, 4, 5)).astype(float)\n        got = np.array(need(ns, \"running_average\")(F.tolist(), 0.1))\n        B = F[0].copy(); want = [B.copy()]\n        for f in F[1:]:\n            B = 0.9 * B + 0.1 * f; want.append(B.copy())\n        close(got, np.array(want), atol=1e-9, what=\"B_t\")\n        m = np.array(need(ns, \"ema_masks\")(F.tolist(), 0.1, 30))\n        ensure(m.shape == (5, 4, 5) and np.array_equal(m, (np.abs(F[1:] - np.array(want)[:-1]) > 30).astype(int)), \"maskeler önceki B'ye göre\")\n    R.test(\"running_average ve ema_masks\", ra)\n",
 "c": "import numpy as np\nfrom skimage import data\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef video(T=60, seed=0, sigma=6):\n    rng = np.random.default_rng(seed)\n    bg = data.camera()[::3, ::3][:120, :160].astype(np.float64)\n    obj = data.coins()[40:80, 40:80].astype(np.float64)\n    frames, gts = [], []\n    for t in range(T):\n        f = bg.copy(); gt = np.zeros(bg.shape, bool)\n        x = 2 * (t - 5)\n        if t >= 5 and x + 30 < 160:\n            f[40:70, x:x + 30] = obj[:30, :30]; gt[40:70, x:x + 30] = True\n        y2 = 104 - 2 * (t - 8)\n        if t >= 8 and 0 <= y2 and y2 + 16 <= 120:\n            f[y2:y2 + 16, 120:136] = 230; gt[y2:y2 + 16, 120:136] = True\n        frames.append(np.clip(f + rng.normal(0, sigma, f.shape), 0, 255).astype(np.uint8)); gts.append(gt)\n    return np.stack(frames), np.stack(gts)\n\n\ndef f1(pred, gt):\n    tp = (pred & gt).sum(); fp = (pred & ~gt).sum(); fn = (~pred & gt).sum()\n    return 2 * tp / max(2 * tp + fp + fn, 1)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"piksel döngüsü yok\", lambda: (max_loops(src, 0, func=\"vibe_init\"), max_loops(src, 0, func=\"vibe_step\")))\n    frames, gts = video()\n\n    def init():\n        S = need(ns, \"vibe_init\")(frames[0], 20, 0)\n        ensure(S.shape == (20, 120, 160), \"(N, H, W)\")\n        P = np.pad(frames[0].astype(int), 1, mode=\"edge\")\n        win = np.lib.stride_tricks.sliding_window_view(P, (3, 3)).reshape(120, 160, 9)\n        ensure(all((win == S[k][..., None]).any(-1).all() for k in range(20)), \"her örnek 3 × 3 komşuluktan gelmeli\")\n    R.test(\"vibe_init\", init)\n\n    def upd():\n        rng = np.random.default_rng(5)\n        f = np.full((60, 80), 120, np.uint8)\n        S = np.full((20, 60, 80), 118, np.int16)\n        changed = []\n        for _ in range(40):\n            before = S.copy()\n            fg = need(ns, \"vibe_step\")(S, f, rng)\n            ensure(not fg.any(), \"her şey arka plan olmalı\")\n            changed.append((before != S).sum() / f.size)\n        rate = float(np.mean(changed))\n        ensure(abs(rate - 2 / 16) < 0.03, f\"kare başına değişen örnek / piksel {rate:.3f}, ≈ 2/φ = 0.125 bekleniyordu\")\n        S2 = np.full((20, 60, 80), 118, np.int16)\n        fg = ns[\"vibe_step\"](S2, np.full((60, 80), 250, np.uint8), rng)\n        ensure(fg.all() and (S2 == 118).all(), \"ön plan pikselleri hiçbir şeyi güncellememeli\")\n        return f\"güncelleme oranı {rate:.3f} (beklenen 0.125: kendi + komşu)\"\n    R.test(\"vibe_step güncelleme kuralları\", upd)\n\n    def run():\n        m = need(ns, \"vibe_run\")(frames)\n        sc = f1(m[10:], gts[10:])\n        naive = np.abs(frames[10:].astype(int) - frames[0].astype(int)) > 25\n        ensure(sc > 0.75, f\"F1 {sc:.3f}\")\n        R.table(\"Sentetik video (σ = 6 gürültü, iki nesne)\", [\"yöntem\", \"F1 (kare 10–59)\"], [[\"|I_t − I_0| > 25\", f\"{f1(naive, gts[10:]):.3f}\"], [\"ViBe\", f\"{sc:.3f}\"]])\n    R.test(\"vibe_run\", run)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef video(T=80, seed=0, sigma=6):\n    rng = np.random.default_rng(seed)\n    bg = data.camera()[::3, ::3][:120, :160].astype(np.float64)\n    obj = data.coins()[40:80, 40:80].astype(np.float64)\n    frames, gts = [], []\n    for t in range(T):\n        f = bg.copy(); gt = np.zeros(bg.shape, bool)\n        f = f + 4 * np.sin(np.arange(160) / 6 + t / 3)[None, :] * (np.arange(120) < 20)[:, None]\n        x = 5 + 2 * t\n        if 30 <= t and x + 30 < 160:\n            f[40:70, x:x + 30] = obj[:30, :30]; gt[40:70, x:x + 30] = True\n        frames.append(np.clip(f + rng.normal(0, sigma, f.shape), 0, 255).astype(np.uint8)); gts.append(gt)\n    return np.stack(frames), np.stack(gts)\n\n\ndef f1(pred, gt):\n    tp = (pred & gt).sum(); fp = (pred & ~gt).sum(); fn = (~pred & gt).sum()\n    return 2 * tp / max(2 * tp + fp + fn, 1)\n\n\ndef check(ns, R):\n    frames, gts = video()\n\n    def unit():\n        g = need(ns, \"GMM\", \"sınıf\")((4, 5))\n        f = torch.full((4, 5), 100.0, dtype=torch.float64)\n        ensure(not g.apply(f).any(), \"ilk kare arka plan\")\n        for _ in range(5):\n            ensure(not g.apply(f + 3).any(), \"küçük değişim arka plan\")\n        ensure(bool(g.apply(f + 120).all()), \"büyük değişim ön plan\")\n        ensure(torch.allclose(g.w.sum(0), torch.ones(4, 5, dtype=torch.float64)), \"ağırlıklar normalize\")\n    R.test(\"GMM birim davranışı\", unit)\n\n    def vid():\n        g = ns[\"GMM\"](frames.shape[1:], alpha=0.02)\n        pred = np.stack([g.apply(torch.tensor(f, dtype=torch.float64)).numpy() for f in frames])\n        mog = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=16, detectShadows=False)\n        pm = np.stack([mog.apply(f) > 0 for f in frames])\n        a, b = f1(pred[30:], gts[30:]), f1(pm[30:], gts[30:])\n        ensure(a > 0.8, f\"F1 {a:.3f}\")\n        R.table(\"Sentetik video (üst şeritte dalgalanan arka plan, nesne 30. karede giriyor)\", [\"yöntem\", \"F1 (kare 30+)\"], [[\"senin GMM (K = 3)\", f\"{a:.3f}\"], [\"OpenCV MOG2\", f\"{b:.3f}\"]])\n    R.test(\"videoda GMM\", vid)\n",
 "e": "import numpy as np\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef ref(frames, alpha, var0=100.0):\n    B = [frames[0]]; V = [torch.full_like(frames[0], var0)]\n    for t in range(1, len(frames)):\n        V.append((1 - alpha) * V[-1] + alpha * (frames[t] - B[-1]) ** 2)\n        B.append((1 - alpha) * B[-1] + alpha * frames[t])\n    return torch.stack(B), torch.stack(V)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız parça döngüsü\", lambda: max_loops(src, 1, func=\"ema_scan\"))\n    g = torch.Generator().manual_seed(0)\n    F = torch.rand(300, 24, 32, generator=g, dtype=torch.float64) * 255\n\n    def sc():\n        Bw, Vw = ref(F, 0.05)\n        for L in (1, 7, 64, 300):\n            close(need(ns, \"ema_scan\")(F, 0.05, L).numpy(), Bw.numpy(), atol=1e-8, what=f\"L = {L}\")\n        B, V = need(ns, \"ema_var_scan\")(F, 0.05)\n        close(V.numpy(), Vw.numpy(), atol=1e-6, what=\"varyans\")\n    R.test(\"ema_scan = döngülü özyineleme\", sc)\n\n    def big():\n        try:\n            out = ns[\"ema_scan\"](F[:, :2, :2].repeat(10, 1, 1), 0.2, 64)\n            ensure(torch.isfinite(out).all(), \"uzun videoda taşma (parçalı tarama gerekli)\")\n        except Exception as e:\n            raise AssertionError(f\"uzun video: {e}\")\n    R.test(\"3000 kare, α = 0.2: taşma yok\", big)\n\n    def m():\n        frames = torch.full((50, 10, 10), 100.0, dtype=torch.float64) + torch.randn(50, 10, 10, generator=g, dtype=torch.float64) * 3\n        frames[30:, 4:6, 4:6] = 200\n        mm = need(ns, \"masks\")(frames, 0.05, 3.0)\n        ensure(bool(mm[30, 4:6, 4:6].all()), \"nesne ön plan\")\n        ensure(mm[10:30].double().mean() < 0.05, \"durağan sahne çoğunlukla arka plan\")\n    R.test(\"masks\", m)\n    if \"ema_scan\" not in ns:\n        return\n    V = torch.rand(200, 240, 320, dtype=torch.float64) * 255\n    t1 = bench(lambda: ref(V, 0.05), repeat=1, number=1)\n    t2 = bench(lambda: ns[\"ema_var_scan\"](V, 0.05), repeat=3)\n    R.table(\"200 × 240 × 320, B ve V\", [\"yöntem\", \"süre\"], [[\"kare döngüsü\", fmt_time(t1)], [\"parçalı tarama (L = 64)\", fmt_time(t2)]],\n            note=\"CPU'da fark küçük olabilir (ikisi de bellek sınırlı); GPU'da kare döngüsü 200 küçük çekirdek başlatır, tarama 4 büyük çekirdek.\")\n",
 "x1": "import numpy as np\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n    bg = data.camera()[::3, ::3][:120, :160].astype(np.float64)\n    tex = data.coins()[60:90, 60:100].astype(np.float64)\n    fr = []\n    for t in range(3):\n        f = bg.copy(); f[50:80, 20 + 15 * t:60 + 15 * t] = tex\n        fr.append(np.clip(f + rng.normal(0, 5, f.shape), 0, 255).astype(np.uint8))\n\n    def t():\n        d2 = need(ns, \"diff2\")(fr[0], fr[1], 30)\n        d3 = need(ns, \"diff3\")(fr[0], fr[1], fr[2], 30)\n        obj = np.zeros(bg.shape, bool); obj[50:80, 35:75] = True\n        ensure((d2 & ~obj).sum() > (d3 & ~obj).sum(), \"üç kare farkı eski konumdaki 'kenar'ı atmalı\")\n        c = need(ns, \"clean\")(d3)\n        b = need(ns, \"boxes\")(c)\n        ensure(len(b) >= 1, \"en az bir kutu\")\n        x, y, w, h = b[0]\n        cx, cy = x + w / 2, y + h / 2\n        ensure(35 <= cx <= 75 and 50 <= cy <= 80, f\"kutu {b[0]} ortadaki karedeki nesnenin üzerinde olmalı\")\n        return f\"iki kare farkı {int(d2.sum())} piksel, üç kare {int(d3.sum())}, temizlik + kutu: {b[0]}\"\n    R.test(\"fark, temizlik, kutu\", t)\n",
 "x2": "import numpy as np\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n    bg = data.camera()[::4, ::4][:96, :128].astype(np.float64)\n    first = bg.copy(); first[30:62, 40:80] = 250\n    frames = np.stack([first] + [bg] * 300)\n    frames = np.clip(frames + rng.normal(0, 4, frames.shape), 0, 255).astype(np.uint8)\n    gm = np.zeros(bg.shape, bool); gm[30:62, 40:80] = True\n\n    def t():\n        rows = []\n        hl = {}\n        for phi in (4, 16, 64):\n            c = need(ns, \"ghost_curve\")(frames, gm, phi)\n            ensure(len(c) == 300 and c[0] > 0.8, \"ilk karelerde hayalet bölgesi ön plan olmalı\")\n            hl[phi] = need(ns, \"half_life\")(c)\n            rows.append([phi, f\"{c[0]:.2f}\", f\"{c[50]:.2f}\", f\"{c[150]:.2f}\", f\"{c[-1]:.2f}\", hl[phi]])\n        ensure(hl[4] < hl[16] < hl[64] or hl[64] == 300, f\"φ büyüdükçe hayalet daha yavaş sönmeli: {hl}\")\n        R.table(\"Hayalet bölgesinde ön plan oranı (40 × 32 piksellik nesne 1. karede gidiyor)\", [\"φ\", \"kare 1\", \"kare 51\", \"kare 151\", \"kare 300\", \"yarılanma\"], rows,\n                note=\"Yayılım hayaleti kenarlardan içeri doğru emer: süre yaklaşık φ × (nesne yarı genişliği) ile ölçeklenir.\")\n    R.test(\"hayalet sönümlemesi\", t)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    bg = cv2.resize(data.coffee(), (200, 150))\n    img = bg.copy()\n    img[40:100, 30:80] = (np.array([20, 120, 220]) * np.ones((60, 50, 1))).astype(np.uint8)\n    sh = np.zeros(bg.shape[:2], bool); sh[90:130, 80:160] = True\n    img[sh] = (img[sh].astype(np.float64) * 0.6).astype(np.uint8)\n    obj = np.zeros_like(sh); obj[40:100, 30:80] = True\n    fg = obj | sh\n\n    def t():\n        m = need(ns, \"shadow_mask\")(img, bg, fg)\n        rec = (m & sh & ~obj).sum() / (sh & ~obj).sum()\n        fa = (m & obj).sum() / obj.sum()\n        ensure(rec > 0.8 and fa < 0.1, f\"gölge yakalama {rec:.2f}, nesneyi gölge sanma {fa:.2f}\")\n        return f\"gölge piksellerinin %{rec * 100:.0f}'ı gölge, nesne piksellerinin %{fa * 100:.0f}'ı yanlışlıkla gölge\"\n    R.test(\"shadow_mask\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_18_background as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Arka plan çıkarma:** ön plan (hareket eden, ilgi çeken) ile arka plan piksellerini ayırmak; gözetim, trafik, hareket tabanlı anomali tespiti için ön işleme bloğu. Zorluklar: gürültü, aydınlatma değişimi, dalgalanan arka plan (yaprak, su), kamera titreşimi, gölgeler, duran nesneler ve **hayaletler** (ilk karede bulunup sonra giden nesnenin bıraktığı yanlış ön plan).

**Basit modeller:** kare farkı $|I_t-I_{t-1}|>T$ (iç kısımları kaçırır, iki kenar verir); üç kare farkı; **üstel hareketli ortalama** $B_t=(1-\alpha)B_{t-1}+\alpha I_t$, ön plan $|I_t-B_{t-1}|>k\sigma$; zamansal medyan.

**Stauffer–Grimson GMM:** her piksel $K$ (3–5) Gauss'luk karışım. Yeni değer, sırayla ($w/\sigma$ büyükten küçüğe) $|x-\mu_k|<2.5\sigma_k$ olan ilk bileşenle eşleşir; eşleşenin ağırlığı artar, ortalama ve varyansı güncellenir; eşleşme yoksa en zayıf bileşen $x$ merkezli geniş bir bileşenle değiştirilir. Arka plan: sıralamada birikimli ağırlığı $T$'yi geçene kadarki bileşenler. OpenCV MOG2 bunun $K$'yı uyarlayan sürümüdür.

**ViBe** (Barnich & Van Droogenbroeck): **parametresiz**, piksel başına geçmiş değerlerden $N\approx20$ örnek.

- *Başlatma:* tek kare yeter — her örnek pikselin rastgele bir 8-komşusundan.
- *Sınıflandırma:* yarıçap $R\approx20$ içinde en az $\#_{min}=2$ örnek varsa arka plan.
- *Güncelleme (yalnız arka plan pikselleri):* $1/\phi$ ($\phi=16$) olasılıkla **rastgele** bir örneği şimdiki değerle değiştir (hafızasız: örnek ömrü üstel dağılır); yine $1/\phi$ olasılıkla rastgele bir **komşunun** rastgele bir örneğini değiştir (uzamsal yayılım: hayaletleri emer, titreşime dayanıklılık).

Artıları: çok hızlı, tamamen paralel sınıflandırma, az son işleme, hayaletleri kendiliğinden siler. Eksileri: dinamik aydınlatma ve kayan arka planlara sabit $R$ ile duyarlılık.

## 1 · Elle

<!-- çözüm -->

Örneklerin farkları 10, 20, 15, 70, 30; yalnız 10 ve 15 yarıçapın içinde: 2 ≥ #min → arka plan.

Hayatta kalma: her arka plan karesinde $1/\phi$ olasılıkla bir güncelleme, her güncellemede belirli bir örneğin silinme olasılığı $1/N$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**ViBe: örnekler $[100,130,95,180,140]$, piksel 110, $R=20$ ($|v-s|<R$), $\#_{min}=2$**

- `cnt`: Yarıçap içindeki örnek sayısı
- `bg`: Arka plan mı? (evet 1, hayır 0)
- `surv`: $N=20$, $\phi=16$: bir örneğin 100 kare sonra hâlâ modelde olma olasılığı $((N-1)/N)^{dt/\phi}$

**Diğer modeller**

- `ema`: Hareketli ortalama $\alpha=0.05$: $B=100$, $I=200$ → yeni $B$
- `gm`: GMM: $\mu=100$, $\sigma=10$, $x=120$: $|x-\mu|/\sigma$
- `tf`: Üç kare farkı, $T=30$: $I_{t-1}=50$, $I_t=120$, $I_{t+1}=60$ → ön plan mı? (1/0)

In [ ]:
cevaplar = dict(
    cnt=None,
    bg=None,
    surv=None,
    ema=None,
    gm=None,
    tf=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"cnt":[2,0],"bg":[1,0],"surv":[0.725726,0.0001],"ema":[105,0],"gm":[2,0],"tf":[1,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python (liste listeleri; örnekler `samples[k][y][x]`).

- `vibe_classify(samples, frame, R=20, nmin=2)` — `|v − s| < R` olan örnek sayısı `≥ nmin` ise arka plan (0), değilse ön plan (1). Sayarken `nmin`'e ulaşınca dur (ViBe'nin hızlandırması; sonuç aynı).
- `running_average(frames, alpha)` — `B_0 = frames[0]`, `B_t = (1−α)B_{t−1} + αI_t`; tüm `B_t` listesi.
- `ema_masks(frames, alpha, T)` — `|I_t − B_{t−1}| > T` maskeleri (t ≥ 1).

In [ ]:
%%writefile stage_b.py
def vibe_classify(samples, frame, R=20, nmin=2):
    ...


def running_average(frames, alpha):
    ...


def ema_masks(frames, alpha, T):
    ...


print(vibe_classify([[[100]], [[130]], [[95]], [[180]], [[140]]], [[110]]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy ViBe (gri seviye, `int16` örnekler). Rastgelelik `rng = np.random.default_rng(seed)`; çağrı sırası serbest (kontrol istatistiksel).

- `vibe_init(frame, N=20, seed=0)` — `(N, H, W)`: her örnek, pikselin 3 × 3 komşuluğundan rastgele bir değer (`edge` dolgu; merkez de seçilebilir).
- `vibe_step(samples, frame, rng, R=20, nmin=2, phi=16)` — sınıflandır (`(np.abs(samples − frame) < R).sum(0) ≥ nmin`), `fg` maskesini döndür; **yerinde** güncelle: arka plan piksellerinin `1/φ`'sinde rastgele bir örnek ← piksel değeri; yine arka plan piksellerinin `1/φ`'sinde rastgele bir 8-komşunun (görüntüye kırpılmış) rastgele örneği ← piksel değeri. Gelişmiş indeksleme ile, piksel döngüsü yok.
- `vibe_run(frames, **kw)` — tüm video için ön plan maskeleri `(T, H, W)` (ilk kare başlatma, maskesi 0). Kare döngüsü serbest.

In [ ]:
%%writefile stage_c.py
import numpy as np


def vibe_init(frame, N=20, seed=0):
    ...


def vibe_step(samples, frame, rng, R=20, nmin=2, phi=16):
    ...


def vibe_run(frames, N=20, R=20, nmin=2, phi=16, seed=0):
    ...


f = np.full((20, 30), 100, np.uint8)
print(vibe_run(np.stack([f, f, f + 80])).sum((1, 2)))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Stauffer–Grimson GMM (PyTorch, piksel başına `K` bileşen, gri seviye, `float64`).

- `class GMM(shape, K=3, alpha=0.01, var0=225.0, T=0.7, w0=0.05)` — durum: `w, mu, var` `(K, H, W)`; başlangıçta ağırlıklar 0 (ilk kare ilk bileşeni `w=1, mu=I_0` yapar).
- `apply(frame)` → ön plan maskesi `(H, W)` bool:
  1. Bileşenleri $w/\sigma$'ya göre azalan sırala (`argsort` + `gather`).
  2. Eşleşme: sıradaki ilk $|x-\mu|<2.5\sigma$ (ve $w>0$) bileşen.
  3. Arka plan bileşenleri: birikimli ağırlığı (sıralı) $T$'yi ilk geçene kadarki dahil; eşleşen bileşen arka plandaysa piksel arka plan.
  4. Güncelleme: $w\leftarrow(1-\alpha)w+\alpha M$; eşleşen için $\rho=\alpha$, $\mu\leftarrow\mu+\rho(x-\mu)$, $\sigma^2\leftarrow\sigma^2+\rho((x-\mu)^2-\sigma^2)$; eşleşme yoksa en son sıradaki bileşen ← $(w_0, x, \text{var0})$; ağırlıkları normalize et.

Kontrol, sentetik videoda F-ölçüsünü ViBe ve `cv2.createBackgroundSubtractorMOG2` ile karşılaştırır.

In [ ]:
%%writefile stage_d.py
import numpy as np
import torch


class GMM:
    def __init__(self, shape, K=3, alpha=0.01, var0=225.0, T=0.7, w0=0.05):
        ...

    def apply(self, frame):
        ...


g = GMM((4, 5))
f = torch.full((4, 5), 100.0, dtype=torch.float64)
print(g.apply(f), g.apply(f + 120))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Üstel hareketli ortalama **doğrusal bir özyinelemedir**: $B_t=aB_{t-1}+bI_t$, $a=1-\alpha$, $b=\alpha$. Kapalı form: $B_t=a^t\big(B_0+\sum_{k=1}^{t}a^{-k}bI_k\big)$ → zaman ekseninde `cumsum`. Uzun videolarda $a^{-k}$ taşar; çözüm **parçalı tarama**: $L$'lik parçalarda yerel `cumsum`, parçalar arası tek bir taşıma (carry) — M10.6'daki paralel tarama fikrinin aynısı.

- `ema_scan(frames, alpha, L=64)` — `frames (T, H, W)` `float64` tensör → `(T, H, W)` $B_t$ ($B_0=I_0$); parça döngüsü (`T/L` tekrar) serbest, kare döngüsü yok.
- `ema_var_scan(frames, alpha, L=64)` — ayrıca $V_t=(1-\alpha)V_{t-1}+\alpha(I_t-B_{t-1})^2$, $V_0=\text{var0}=100$; `(B, V)`. (İpucu: önce B'yi tara, sonra $(I_t-B_{t-1})^2$ dizisini aynı tarayıcıyla.)
- `masks(frames, alpha, k=3.0)` — $|I_t-B_{t-1}|>k\sqrt{V_{t-1}}$, `t ≥ 1` (ilk kare 0).

Not: **seçici** güncelleme (yalnız arka plan piksellerini güncellemek) özyinelemeyi doğrusal olmayan yapar ve bu hileyi bozar.

In [ ]:
%%writefile stage_e.py
import torch


def ema_scan(frames, alpha, L=64):
    ...


def ema_var_scan(frames, alpha, L=64, var0=100.0):
    ...


def masks(frames, alpha, k=3.0):
    ...


x = torch.rand(10, 4, 4, dtype=torch.float64)
print(ema_scan(x, 0.1, L=4).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Kare farkı, morfolojik temizlik ve blob kutuları

- `diff2(prev, cur, T)` — iki kare farkı maskesi.
- `diff3(prev, cur, nxt, T)` — üç kare farkı: iki farkın **ve**'si (ortadaki karedeki nesneyi verir, iki kenar yerine).
- `clean(mask, k=3)` — açma sonra kapama (`cv2.MORPH_ELLIPSE` k × k).
- `boxes(mask, min_area=20)` — `cv2.connectedComponentsWithStats` (8-bağlantı) ile alanı `≥ min_area` bileşenlerin `(x, y, w, h)` listesi, alana göre azalan.

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def diff2(prev, cur, T):
    ...


def diff3(prev, cur, nxt, T):
    ...


def clean(mask, k=3):
    ...


def boxes(mask, min_area=20):
    ...


print("fark")

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · ViBe'de hayalet sönümlemesi ve φ'nin etkisi

Hayalet deneyi: ilk karede bir nesne var, ikinci karede gidiyor. ViBe'nin modeli nesneyi arka plan sanıyor; açığa çıkan gerçek arka plan **hayalet** olarak ön planda görünür ve uzamsal yayılımla zamanla emilir.

- `ghost_curve(frames, ghost_mask, phi, seed=0)` — her karede hayalet bölgesindeki ön plan oranı (liste). c'deki `vibe_init` / `vibe_step` yardımcıları hazır verildi.
- `half_life(curve)` — oran ilk değerin yarısının altına ilk indiği kare indisi (inmezse `len(curve)`).

In [ ]:
%%writefile extra_x2.py
import numpy as np

OFF = np.array([(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)])


def vibe_init(frame, N=20, seed=0):
    rng = np.random.default_rng(seed)
    H, W = frame.shape
    P = np.pad(frame.astype(np.int16), 1, mode="edge")
    yy, xx = np.mgrid[:H, :W]
    dy, dx = rng.integers(-1, 2, (2, N, H, W))
    return P[yy + 1 + dy, xx + 1 + dx]


def vibe_step(samples, frame, rng, R=20, nmin=2, phi=16):
    N, H, W = samples.shape
    f = frame.astype(np.int16)
    fg = (np.abs(samples - f) < R).sum(0) < nmin
    bg = ~fg
    y, x = np.nonzero(bg & (rng.random((H, W)) < 1 / phi))
    samples[rng.integers(0, N, len(y)), y, x] = f[y, x]
    y, x = np.nonzero(bg & (rng.random((H, W)) < 1 / phi))
    o = OFF[rng.integers(0, 8, len(y))]
    ny, nx = np.clip(y + o[:, 0], 0, H - 1), np.clip(x + o[:, 1], 0, W - 1)
    samples[rng.integers(0, N, len(y)), ny, nx] = f[y, x]
    return fg


def ghost_curve(frames, ghost_mask, phi, seed=0):
    ...


def half_life(curve):
    ...


print("hayalet")

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Gölge tespiti (HSV'de parlaklık oranı)

Cucchiara ve ark. (2003) gölge kuralı (HSV, $V$ 0–1, $H$ derece, $S$ 0–1): ön plan pikseli, arka planla karşılaştırıldığında

$$\alpha\le\frac{V_I}{V_B}\le\beta,\quad|S_I-S_B|\le\tau_S,\quad\min(|H_I-H_B|,360-|H_I-H_B|)\le\tau_H$$

ise **gölgedir** (koyulaşmış ama renk ve doygunluğu aynı arka plan).

- `shadow_mask(img, bg, fg, alpha=0.4, beta=0.93, tS=0.15, tH=40)` — `img`, `bg` RGB `uint8`; HSV `cv2.cvtColor(…, cv2.COLOR_RGB2HSV_FULL)` (H 0–255 → derece ×360/256, S, V /255); yalnız `fg` içindeki pikseller için. Bool maske.

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def shadow_mask(img, bg, fg, alpha=0.4, beta=0.93, tS=0.15, tH=40):
    ...


print("gölge")

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: ViBe'yi CDnet 2014'te değerlendir

[Barnich & Van Droogenbroeck — ViBe: A Universal Background Subtraction Algorithm for Video Sequences (IEEE TIP 2011)](https://doi.org/10.1109/TIP.2010.2101613)

c aşamasındaki ViBe'yi renkli videolara (RGB'de L1 uzaklık) ve tam makale sürümüne genişlet; CDnet 2014'ün baseline, dynamicBackground, cameraJitter, shadow ve intermittentObjectMotion kategorilerinde F-ölçüsü hesapla ve OpenCV MOG2 ile KNN'i karşılaştır. "Experiments and improvements" makalesindeki iyileştirmelerden en az ikisini (ör. güncelleme maskesi ile yayılımı ayırmak, uyarlamalı R) ekle.


**Adımlar**

1. CDnet'in ROI ve 'unknown' piksellerini dışlayan değerlendirme betiği (Re, Pr, F, PWC).
2. φ, N, R, #min için parametre taraması; kategori bazında en iyi F.
3. Medyan süzgeç + morfolojik kapama son işleme.

**Kabul ölçütleri**

- Beş kategoride F-ölçüsü tablosu (ViBe, ViBe+iyileştirmeler, MOG2, KNN).
- cameraJitter'da uzamsal yayılım kapalı/açık ablasyonu.

## Makale ödevi: RT-SBS: anlamsal bilgiyle gerçek zamanlı arka plan çıkarma

[Cioppa, Braham, Van Droogenbroeck — Real-Time Semantic Background Subtraction (ICIP 2020)](https://arxiv.org/abs/2002.04993)

Her karede değil, her k karede bir çalışan hafif bir anlamsal bölütleme ağının (ör. önceden eğitilmiş DeepLab/LR-ASPP; M4.10) ön plan olasılığını ViBe maskesiyle birleştiren karar kurallarını uygula; aradaki karelerde anlamsal bilgiyi ViBe'nin değişim bilgisiyle "taşı". CDnet'te ViBe ve her kare anlamsal sürümle F-ölçüsü / FPS karşılaştır.


**Adımlar**

1. Anlamsal ön plan olasılığı: insan, araç gibi sınıfların olasılık toplamı; arka plan anlamsal modeli.
2. Karar kuralları: iki eşikle (τ_BG, τ_FG) arka plan/ön plan zorlama, aksi halde ViBe.
3. Anlamsal ağın çalışmadığı karelerde son kararların piksel başına yeniden kullanımı.

**Kabul ölçütleri**

- k = 1, 2, 5 için F ve FPS tablosu; anlamsal bilginin hayalet ve gölge hatalarını azalttığını gösteren örnekler.

## Kaynaklar

- [Barnich & Van Droogenbroeck — ViBe: A Universal Background Subtraction Algorithm for Video Sequences (IEEE TIP 2011)](https://doi.org/10.1109/TIP.2010.2101613) — ©
- [Stauffer & Grimson — Adaptive background mixture models for real-time tracking (CVPR 1999)](https://doi.org/10.1109/CVPR.1999.784637) — ©
- [Zivkovic — Improved adaptive Gaussian mixture model for background subtraction (ICPR 2004) — OpenCV MOG2](https://doi.org/10.1109/ICPR.2004.1333992) — ©
- [Van Droogenbroeck & Paquot — Background subtraction: Experiments and improvements for ViBe (CVPRW 2012)](https://doi.org/10.1109/CVPRW.2012.6238924) — ©